## 사전작업

In [1]:
!pip install xgboost lightgbm missingno

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 57.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 52.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [missingno]/4 [lightgbm]


In [2]:
import warnings
warnings.filterwarnings("ignore")

import os
from os.path import join
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


## 데이터 경로 설정

In [27]:
data_dir = '~/work/AIFFEL_quest_eng/Main_Quest/Quest01'

train_data_path = join(data_dir, 'train.csv')
test_data_path = join(data_dir, 'test.csv')
submission_path = join(data_dir, "sample_submission.csv")

train = pd.read_csv(train_data_path)
test = pd.read_csv(test_data_path)


[LightGBM] [Info] Total Bins 2298
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[LightGBM] [Info] Start training from score 13.035255
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015776 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2298
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[Lig

## 데이터 읽기

In [4]:
train = pd.read_csv(train_data_path)
train_raw = pd.read_csv(train_data_path)
test = pd.read_csv(test_data_path)

print(train.shape)
print(test.shape)
print(train.columns)
train.head()

(15035, 21)
(6468, 20)
Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,1,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
2,2,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
3,3,20140627T000000,257500.0,3,2.25,1715,6819,2.0,0,0,...,7,1715,0,1995,0,98003,47.3097,-122.327,2238,6819
4,4,20150115T000000,291850.0,3,1.50,1060,9711,1.0,0,0,...,7,1060,0,1963,0,98198,47.4095,-122.315,1650,9711


## date 전처리

In [5]:
train["date"] = train["date"].apply(lambda x: x[:6]).astype(int)
test["date"] = test["date"].apply(lambda x: x[:6]).astype(int)

print(train.info())
print()
print(train.isnull().sum())
print()
print(test.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15035 entries, 0 to 15034
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             15035 non-null  int64  
 1   date           15035 non-null  int64  
 2   price          15035 non-null  float64
 3   bedrooms       15035 non-null  int64  
 4   bathrooms      15035 non-null  float64
 5   sqft_living    15035 non-null  int64  
 6   sqft_lot       15035 non-null  int64  
 7   floors         15035 non-null  float64
 8   waterfront     15035 non-null  int64  
 9   view           15035 non-null  int64  
 10  condition      15035 non-null  int64  
 11  grade          15035 non-null  int64  
 12  sqft_above     15035 non-null  int64  
 13  sqft_basement  15035 non-null  int64  
 14  yr_built       15035 non-null  int64  
 15  yr_renovated   15035 non-null  int64  
 16  zipcode        15035 non-null  int64  
 17  lat            15035 non-null  float64
 18  long  

## 타깃값 분리

In [6]:
y = train["price"]
del train["price"]

print(y.head())
print(train.columns)

0    221900.0
1    180000.0
2    510000.0
3    257500.0
4    291850.0
Name: price, dtype: float64
Index(['id', 'date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
       'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
       'sqft_living15', 'sqft_lot15'],
      dtype='object')


## id 제거

In [7]:
del train["id"]
print(train.columns)

test_id = test["id"]
del test["id"]
print(test.columns)

Index(['date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
       'sqft_living15', 'sqft_lot15'],
      dtype='object')
Index(['date', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long',
       'sqft_living15', 'sqft_lot15'],
      dtype='object')


## 타깃 로그 변환

In [8]:
y = np.log1p(y)
print(y.head())

0    12.309987
1    12.100718
2    13.142168
3    12.458779
4    12.583999
Name: price, dtype: float64


## 전처리 결과 확인

In [9]:
print(train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15035 entries, 0 to 15034
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           15035 non-null  int64  
 1   bedrooms       15035 non-null  int64  
 2   bathrooms      15035 non-null  float64
 3   sqft_living    15035 non-null  int64  
 4   sqft_lot       15035 non-null  int64  
 5   floors         15035 non-null  float64
 6   waterfront     15035 non-null  int64  
 7   view           15035 non-null  int64  
 8   condition      15035 non-null  int64  
 9   grade          15035 non-null  int64  
 10  sqft_above     15035 non-null  int64  
 11  sqft_basement  15035 non-null  int64  
 12  yr_built       15035 non-null  int64  
 13  yr_renovated   15035 non-null  int64  
 14  zipcode        15035 non-null  int64  
 15  lat            15035 non-null  float64
 16  long           15035 non-null  float64
 17  sqft_living15  15035 non-null  int64  
 18  sqft_l

상관관계 높은 컬럼 전체 보기

In [10]:
corr = train_raw.corr(numeric_only=True)
print(corr["price"].sort_values(ascending=False))

price            1.000000
sqft_living      0.702899
grade            0.667211
sqft_above       0.608577
sqft_living15    0.586419
bathrooms        0.525479
view             0.400806
bedrooms         0.323672
sqft_basement    0.322218
lat              0.301604
waterfront       0.265738
floors           0.262588
yr_renovated     0.140808
sqft_lot         0.096793
sqft_lot15       0.086384
yr_built         0.047290
condition        0.039740
long             0.023547
id               0.020899
zipcode         -0.051498
Name: price, dtype: float64


왜도 높은 변수 보기

In [11]:
skew_df = train.select_dtypes(include=["int64", "float64"]).skew().sort_values(ascending=False)
# waterfront, view, yr_renovated는 예외(0이 많고, 일부만 특별한 경우)
print(skew_df) 

sqft_lot         13.350500
waterfront       11.728113
sqft_lot15       10.028412
yr_renovated      4.569374
view              3.378768
sqft_basement     1.556555
sqft_living       1.492472
sqft_above        1.429070
sqft_living15     1.125932
condition         1.044110
long              0.917991
date              0.768872
grade             0.751671
floors            0.589420
bedrooms          0.518581
bathrooms         0.513566
zipcode           0.405891
yr_built         -0.469626
lat              -0.488040
dtype: float64


~로그 변환 추가(가장 큰 내부 면적 관련)~ 수치 오히려 상승.. 1602588;;

## RMSE 함수 정의

In [12]:
def rmse(y_test, y_pred):
    return np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))

## 여러 모델 준비

In [13]:
random_state = 2020

gboost = GradientBoostingRegressor(random_state=random_state)
xgboost = XGBRegressor(random_state=random_state)
lightgbm = LGBMRegressor(random_state=random_state)
rdforest = RandomForestRegressor(random_state=random_state)

models = [gboost, xgboost, lightgbm, rdforest]

## 모델별 점수 비교

In [14]:
df = {}

for model in models:
    model_name = model.__class__.__name__

    X_train, X_test, y_train, y_test = train_test_split(
        train, y, random_state=random_state, test_size=0.2
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    df[model_name] = rmse(y_test, y_pred)

df

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001380 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2298
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[LightGBM] [Info] Start training from score 13.047779


{'GradientBoostingRegressor': np.float64(128360.19649691365),
 'XGBRegressor': np.float64(117618.22355411823),
 'LGBMRegressor': np.float64(111920.36735892233),
 'RandomForestRegressor': np.float64(125486.94461618949)}

## 모델 점수 비교 함수

In [15]:
def get_scores(models, train, y):
    df = {}

    for model in models:
        model_name = model.__class__.__name__

        X_train, X_test, y_train, y_test = train_test_split(
            train, y, random_state=random_state, test_size=0.2
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        df[model_name] = rmse(y_test, y_pred)

    score_df = pd.DataFrame(df, index=["RMSE"]).T.sort_values("RMSE", ascending=False)
    return score_df


print(get_scores(models, train, y))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002911 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2298
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[LightGBM] [Info] Start training from score 13.047779
                                    RMSE
GradientBoostingRegressor  128360.196497
RandomForestRegressor      125486.944616
XGBRegressor               117618.223554
LGBMRegressor              111920.367359


## GridSearch 함수

In [16]:
def my_GridSearch(model, train, y, param_grid, verbose=2, n_jobs=5):
    grid_model = GridSearchCV(
        model,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=5,
        verbose=verbose,
        n_jobs=n_jobs,
    )

    grid_model.fit(train, y)

    params = grid_model.cv_results_["params"]
    score = grid_model.cv_results_["mean_test_score"]

    results = pd.DataFrame(params)
    results["score"] = score
    results["RMSLE"] = np.sqrt(-results["score"])
    results = results.sort_values("RMSLE")

    return results

## 하이퍼파라미터 튜닝 범위 설정

- 이 쪽을 바꾸는 게 나을 거 같음

In [17]:
param_grid = {
      "n_estimators": [200, 300], # 트리를 몇 개 쌓을 지(작덜 배울지 더 배울지)
      "max_depth": [8, 10], # 트리의 깊이(단순하게 볼지 복잡하게 볼지)
      "learning_rate": [0.03, 0.05], # 얼마나 빠르게 학습할지(천천히 배울지 조금 빠르게 배울지 / 천천히 배울 수록 과적합 발생 감소)"num_leaves": [20, 31, 40]
      "num_leaves": [20, 31, 40] # 새로운 파라미터, 트리 복잡돋와 관련
}

## LightGBM 튜닝

In [18]:
model = LGBMRegressor(random_state=2020) # 
grid_result = my_GridSearch(
    model, 
    train,
    y, 
    param_grid, 
    verbose=0, 
    n_jobs=5)
print(grid_result)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019407 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2298
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[LightGBM] [Info] Start training from score 13.035255
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014262 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2331
[LightGBM] [Info] Number of data points in the train set: 12028, number of used features: 19
[LightGBM] [Info] Start training from score 13.051670
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014095 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total 

## 최종 모델 학습

In [19]:
model = LGBMRegressor(max_depth=10, n_estimators=300, num_leaves=40, random_state=2020)
model.fit(train, y)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2338
[LightGBM] [Info] Number of data points in the train set: 15035, number of used features: 19
[LightGBM] [Info] Start training from score 13.048122
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

,boosting_type,'gbdt'
,num_leaves,40
,max_depth,10
,learning_rate,0.1
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## 예측값 확인

In [20]:
prediction = model.predict(test)
print(prediction)

[13.08097711 13.11333299 14.13945965 ... 13.08851578 12.74630575
 13.00308335]


## 원래 스케일로 복원

In [21]:
prediction = np.expm1(prediction)
print(prediction)

[ 479728.2269439   495504.13042938 1382575.92771163 ...  483358.40998847
  343280.3842584   443778.61279412]


## 제출 파일 확인

In [22]:
submission = pd.read_csv(submission_path)
print(submission.head())

      id   price
0  15035  100000
1  15036  100000
2  15037  100000
3  15038  100000
4  15039  100000


## 예측값 넣기

In [23]:
submission["price"] = prediction
print(submission.head())

      id         price
0  15035  4.797282e+05
1  15036  4.955041e+05
2  15037  1.382576e+06
3  15038  2.768948e+05
4  15039  3.255771e+05


## 저장

In [24]:
submission_csv_path = "{}/submission_{}_RMSLE_{}.csv".format(data_dir, "lgbm", "0.164399")
submission.to_csv(submission_csv_path, index=False)
print(submission_csv_path)

~/work/AIFFEL_quest_eng/Data_Analysis/DA02/submission_lgbm_RMSLE_0.164399.csv


## 저장 함수

In [25]:
def save_submission(model, train, y, test, model_name, rmsle=None):
      model.fit(train, y)
      prediction = model.predict(test)
      prediction = np.expm1(prediction)

      data_dir = os.path.expanduser('~/work/AIFFEL_quest_eng/Data_Analysis/DA02')
      submission_path = join(data_dir, 'sample_submission.csv')

      submission = pd.read_csv(submission_path)
      submission["price"] = prediction

      submission_csv_path = join(
          data_dir,
          'submission_{}_RMSLE_{}.csv'.format(model_name, rmsle)
      )
      submission.to_csv(submission_csv_path, index=False)

      print(submission.head())
      print('{} saved!'.format(submission_csv_path))

## 함수 실행 예시

In [26]:
save_submission(model, train, y, test, "lgbm", rmsle="0.161913")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001693 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2338
[LightGBM] [Info] Number of data points in the train set: 15035, number of used features: 19
[LightGBM] [Info] Start training from score 13.048122
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain